# Global Earthquake Monitoring & Forecasting

## Notebook 1 - Data Collection & Preprocessing

### Objectives

- Download earthquake data from USGS
- Clean and preprocess the data
- Create useful date features
- Save a processed dataset
- Create a daily time-series dataset

In [1]:
import os
import requests
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [2]:
folders = [
    "data",
    "data/raw",
    "data/processed",
    "outputs",
    "outputs/maps",
    "outputs/figures",
    "outputs/forecasts",
    "outputs/reports"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [7]:
url = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query.csv"
    "?starttime=2025-01-01"
    "&endtime=2025-12-31"
    "&minmagnitude=2.5"
    "&limit=20000"
    "&orderby=time"
)

response = requests.get(url)

with open("data/raw/usgs_earthquakes.csv", "wb") as file:
    file.write(response.content)

print("Dataset downloaded successfully.")

Dataset downloaded successfully.


In [8]:
earthquakes = pd.read_csv(
    "data/raw/usgs_earthquakes.csv"
)

earthquakes.head()

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2025-12-30T23:51:36.674Z,39.4639,142.7562,35.000,4.8,mb,65.0,128.0,1.853,0.85,...,2026-04-01T22:48:43.040Z,"69 km E of Yamada, Japan",earthquake,7.39,1.9090,0.051,121.0,reviewed,us,us
1,2025-12-30T22:36:09.826Z,3.0448,128.8654,10.000,4.5,mb,29.0,95.0,3.772,0.62,...,2026-04-01T22:49:07.040Z,"173 km NNE of Tobelo, Indonesia",earthquake,5.62,1.9190,0.131,17.0,reviewed,us,us
2,2025-12-30T22:25:43.819Z,60.5503,-140.1190,5.000,3.3,ml,39.0,118.0,0.461,0.61,...,2026-05-14T19:42:46.360Z,"113 km N of Yakutat, Alaska",earthquake,3.04,2.0010,0.036,100.0,reviewed,us,us
3,2025-12-30T22:19:46.317Z,39.5984,143.3045,27.735,4.8,mww,62.0,122.0,2.088,0.41,...,2026-04-01T22:48:43.040Z,"117 km E of Miyako, Japan",earthquake,6.26,5.0050,0.103,9.0,reviewed,us,us
4,2025-12-30T22:14:01.822Z,62.7050,-149.2790,59.600,2.5,ml,59.0,27.0,0.200,0.90,...,2026-05-04T07:39:01.337Z,"51 km ENE of Chase, Alaska",earthquake,2.30,2.9538,0.300,36.0,reviewed,ak,ak


In [9]:
print(earthquakes.columns.tolist())

['time', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'nst', 'gap', 'dmin', 'rms', 'net', 'id', 'updated', 'place', 'type', 'horizontalError', 'depthError', 'magError', 'magNst', 'status', 'locationSource', 'magSource']


In [10]:
earthquakes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 22 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   time             20000 non-null  object 
 1   latitude         20000 non-null  float64
 2   longitude        20000 non-null  float64
 3   depth            20000 non-null  float64
 4   mag              20000 non-null  float64
 5   magType          20000 non-null  object 
 6   nst              19233 non-null  float64
 7   gap              19229 non-null  float64
 8   dmin             19226 non-null  float64
 9   rms              20000 non-null  float64
 10  net              20000 non-null  object 
 11  id               20000 non-null  object 
 12  updated          20000 non-null  object 
 13  place            20000 non-null  object 
 14  type             20000 non-null  object 
 15  horizontalError  19005 non-null  float64
 16  depthError       19999 non-null  float64
 17  magError    

In [11]:
earthquakes = earthquakes.drop_duplicates()

print("Remaining records:", len(earthquakes))

Remaining records: 20000


In [12]:
earthquakes["time"] = pd.to_datetime(
    earthquakes["time"],
    format="mixed",
    utc=True,
    errors="coerce"
)

earthquakes = earthquakes.dropna(subset=["time"])

In [13]:
earthquakes = earthquakes.rename(
    columns={
        "time": "Time",
        "latitude": "Latitude",
        "longitude": "Longitude",
        "depth": "Depth_km",
        "mag": "Magnitude",
        "place": "Location"
    }
)

earthquakes.head()

,Time,Latitude,Longitude,Depth_km,Magnitude,magType,nst,gap,dmin,rms,...,updated,Location,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2025-12-30 23:51:36.674000+00:00,39.4639,142.7562,35.000,4.8,mb,65.0,128.0,1.853,0.85,...,2026-04-01T22:48:43.040Z,"69 km E of Yamada, Japan",earthquake,7.39,1.9090,0.051,121.0,reviewed,us,us
1,2025-12-30 22:36:09.826000+00:00,3.0448,128.8654,10.000,4.5,mb,29.0,95.0,3.772,0.62,...,2026-04-01T22:49:07.040Z,"173 km NNE of Tobelo, Indonesia",earthquake,5.62,1.9190,0.131,17.0,reviewed,us,us
2,2025-12-30 22:25:43.819000+00:00,60.5503,-140.1190,5.000,3.3,ml,39.0,118.0,0.461,0.61,...,2026-05-14T19:42:46.360Z,"113 km N of Yakutat, Alaska",earthquake,3.04,2.0010,0.036,100.0,reviewed,us,us
3,2025-12-30 22:19:46.317000+00:00,39.5984,143.3045,27.735,4.8,mww,62.0,122.0,2.088,0.41,...,2026-04-01T22:48:43.040Z,"117 km E of Miyako, Japan",earthquake,6.26,5.0050,0.103,9.0,reviewed,us,us
4,2025-12-30 22:14:01.822000+00:00,62.7050,-149.2790,59.600,2.5,ml,59.0,27.0,0.200,0.90,...,2026-05-04T07:39:01.337Z,"51 km ENE of Chase, Alaska",earthquake,2.30,2.9538,0.300,36.0,reviewed,ak,ak


In [14]:
earthquakes["Date"] = earthquakes["Time"].dt.date
earthquakes["Year"] = earthquakes["Time"].dt.year
earthquakes["Month"] = earthquakes["Time"].dt.month
earthquakes["Day"] = earthquakes["Time"].dt.day
earthquakes["Hour"] = earthquakes["Time"].dt.hour
earthquakes["Weekday"] = earthquakes["Time"].dt.day_name()

In [15]:
earthquakes = earthquakes[
    [
        "Time",
        "Date",
        "Year",
        "Month",
        "Day",
        "Hour",
        "Weekday",
        "Latitude",
        "Longitude",
        "Depth_km",
        "Magnitude",
        "Location"
    ]
]

earthquakes.head()

,Time,Date,Year,Month,Day,Hour,Weekday,Latitude,Longitude,Depth_km,Magnitude,Location
0,2025-12-30 23:51:36.674000+00:00,2025-12-30,2025,12,30,23,Tuesday,39.4639,142.7562,35.000,4.8,"69 km E of Yamada, Japan"
1,2025-12-30 22:36:09.826000+00:00,2025-12-30,2025,12,30,22,Tuesday,3.0448,128.8654,10.000,4.5,"173 km NNE of Tobelo, Indonesia"
2,2025-12-30 22:25:43.819000+00:00,2025-12-30,2025,12,30,22,Tuesday,60.5503,-140.1190,5.000,3.3,"113 km N of Yakutat, Alaska"
3,2025-12-30 22:19:46.317000+00:00,2025-12-30,2025,12,30,22,Tuesday,39.5984,143.3045,27.735,4.8,"117 km E of Miyako, Japan"
4,2025-12-30 22:14:01.822000+00:00,2025-12-30,2025,12,30,22,Tuesday,62.7050,-149.2790,59.600,2.5,"51 km ENE of Chase, Alaska"


In [16]:
earthquakes.describe()

,Year,Month,Day,Hour,Latitude,Longitude,Depth_km,Magnitude
count,20000.0,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000
mean,2025.0,8.501550,16.012000,11.477850,28.146457,2.097588,54.591497,3.949334
std,0.0,2.185781,9.044907,7.013242,31.139010,136.579214,100.523567,0.848348
min,2025.0,5.000000,1.000000,0.000000,-65.608800,-179.994400,-3.370000,2.500000
25%,2025.0,7.000000,8.000000,5.000000,7.359625,-139.753025,10.000000,3.100000
50%,2025.0,8.000000,16.000000,11.000000,39.669050,-27.886300,17.385500,4.200000
75%,2025.0,10.000000,24.000000,18.000000,52.330975,149.341850,51.548000,4.500000
max,2025.0,12.000000,31.000000,23.000000,87.081500,179.999400,669.556000,8.800000


In [17]:
earthquakes.to_csv(
    "data/processed/earthquakes_processed.csv",
    index=False
)

print("Processed dataset saved.")

Processed dataset saved.


In [18]:
daily_counts = (
    earthquakes
    .groupby("Date")
    .size()
    .reset_index(name="Earthquake_Count")
)

daily_counts.head()

,Date,Earthquake_Count
0,2025-05-04,19
1,2025-05-05,63
2,2025-05-06,59
3,2025-05-07,70
4,2025-05-08,67


In [19]:
daily_counts.to_csv(
    "data/processed/earthquake_daily_timeseries.csv",
    index=False
)

print("Time-series dataset saved.")

Time-series dataset saved.


In [20]:
print("Processed Dataset Shape:", earthquakes.shape)
print("Daily Counts Shape:", daily_counts.shape)

earthquakes.head()

Processed Dataset Shape: (20000, 12)
Daily Counts Shape: (241, 2)


,Time,Date,Year,Month,Day,Hour,Weekday,Latitude,Longitude,Depth_km,Magnitude,Location
0,2025-12-30 23:51:36.674000+00:00,2025-12-30,2025,12,30,23,Tuesday,39.4639,142.7562,35.000,4.8,"69 km E of Yamada, Japan"
1,2025-12-30 22:36:09.826000+00:00,2025-12-30,2025,12,30,22,Tuesday,3.0448,128.8654,10.000,4.5,"173 km NNE of Tobelo, Indonesia"
2,2025-12-30 22:25:43.819000+00:00,2025-12-30,2025,12,30,22,Tuesday,60.5503,-140.1190,5.000,3.3,"113 km N of Yakutat, Alaska"
3,2025-12-30 22:19:46.317000+00:00,2025-12-30,2025,12,30,22,Tuesday,39.5984,143.3045,27.735,4.8,"117 km E of Miyako, Japan"
4,2025-12-30 22:14:01.822000+00:00,2025-12-30,2025,12,30,22,Tuesday,62.7050,-149.2790,59.600,2.5,"51 km ENE of Chase, Alaska"
